# Unsupervised Anomaly Detection Prototype (IsolationForest Baseline) - Project 3
> **Exploratory Prototype — Not the Shipped Production Model:** This notebook documents the early Phase-1 unsupervised `IsolationForest` baseline on 4 static/rolling features (which achieved only ~5.5% in-sample recall). Because unsupervised isolation trees underperformed on rare smurfing typologies, production (`src/models/train.py` & `src/models/anomaly_detector.py`) replaced this prototype with a supervised `RandomForestClassifier(n_estimators=50, max_depth=10, class_weight='balanced')` trained on all 7 stateful `StreamingStateManager` features (`CANONICAL_AML_FEATURES`), achieving `26.23%` out-of-sample recall (`32 / 122`) at threshold `0.62`.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, recall_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "SAML-D.csv"
df = pd.read_csv(DATA_PATH, nrows=100000)
df["is_structuring"] = ((df["Amount"] >= 8000.0) & (df["Amount"] < 10000.0)).astype(int)
proto_features = ["Amount", "is_structuring"]
X_proto = df[proto_features]
y_proto = df["Is_laundering"].astype(int)

In [ ]:
# In-sample diagnostic fit of the early IsolationForest prototype
iso = IsolationForest(n_estimators=100, contamination=0.005, random_state=42, n_jobs=-1)
raw_preds = iso.fit_predict(X_proto)
in_sample_preds = (raw_preds == -1).astype(int)
print(
    f"In-Sample Prototype Diagnostic Recall: {recall_score(y_proto, in_sample_preds) * 100:.2f}%"
)
print(classification_report(y_proto, in_sample_preds, digits=4))